# Search Examples

Comprehensive examples of QuranKit's search capabilities with different normalization profiles.

In [ ]:
from qurankit import Quran
from qurankit.search import get_engine, SearchMode

quran = Quran.load()
engine = get_engine()

## Normalization Profiles

QuranKit provides 4 built-in normalization profiles:

| Profile ID | Description | Use Case |
|------------|-------------|----------|
| `exact-source-v1` | Preserves all diacritics, marks, orthography | Exact textual matching |
| `canonical-unicode-v1` | NFC normalization, preserves Quranic marks | Canonical equivalence |
| `diacritics-insensitive-v1` | Removes harakat, keeps Quranic marks | Diacritic-insensitive search |
| `search-broad-v1` | Removes diacritics, normalizes alif/hamza/ya, optional ta marbutah | Broad/fuzzy search |

In [ ]:
# List all available profiles
profiles = engine.list_profiles()
for p in profiles:
    print(f"{p.profile_id}: {p.description}")
    print(f"  Removes diacritics: {p.remove_diacritics}")
    print(f"  Removes Quranic marks: {p.remove_quranic_marks}")
    print(f"  Alif/hamza equivalence: {p.alif_hamza_equivalence}")
    print(f"  Ya/alif maqsurah equivalence: {p.ya_maqsurah_equivalence}")
    print(f"  Ta marbutah -> ha: {p.ta_marbutah_to_ha}")
    print()

## Basic Search

Search using the Quran class with different modes.

In [ ]:
# Search for Bismillah
query = "بسم الله الرحمن الرحيم"

for mode in [SearchMode.EXACT_SOURCE, SearchMode.CANONICAL_UNICODE, 
             SearchMode.DIACRITICS_INSENSITIVE, SearchMode.SEARCH_BROAD]:
    results = quran.search(query, mode=mode)
    print(f"{mode.value}: {len(results)} results")
    for r in results[:2]:
        print(f"  {r.reference}: {r.matched_text[:60]} [{r.match_type.value}]")
    print()

## Direct Normalization Engine Usage

Use the normalization engine directly for character-level mappings.

In [ ]:
text = "بِسْمِ اللَّهِ الرَّحْمَٰنِ الرَّحِيمِ"

for profile_id in ["exact-source-v1", "canonical-unicode-v1", "diacritics-insensitive-v1", "search-broad-v1"]:
    result = engine.normalize(text, profile_id, return_mappings=True)
    print(f"Profile: {profile_id}")
    print(f"  Original:  {text}")
    print(f"  Normalized: {result.normalized}")
    if result.mappings:
        print(f"  Mappings ({len(result.mappings)}):")
        for start, end, orig, norm in result.mappings[:5]:
            if norm:
                print(f"    [{start}:{end}] '{orig}' -> '{norm}'")
            else:
                print(f"    [{start}:{end}] '{orig}' -> (removed)")
        if len(result.mappings) > 5:
            print(f"    ... and {len(result.mappings) - 5} more")
    print()

## Arabic Text Processing Details

Examples of how each profile handles specific Arabic characters.

In [ ]:
test_cases = [
    ("Alif variants", "أ إ آ ا"),
    ("Hamza forms", "ئ ؤ ء"),
    ("Ya/Alif maqsurah", "ى ي"),
    ("Ta marbutah", "ة ه"),
    ("Diacritics", "بِسْمِ اللَّهِ"),
    ("Quranic marks", "۞ ۩ ۝"),
    ("Tatweel", "بِسْمِ اللَّهِ"),
    ("Shadda", "مُحَمَّد"),
]

for name, text in test_cases:
    print(f"{name}: {text}")
    for profile_id in ["exact-source-v1", "diacritics-insensitive-v1", "search-broad-v1"]:
        result = engine.normalize(text, profile_id)
        print(f"  {profile_id}: {result.normalized}")
    print()

## Phrase Search

Search for multi-word phrases.

In [ ]:
# Search for common phrases
phrases = [
    "الْحَمْدُ لِلَّهِ",
    "رَبِّ الْعَالَمِينَ",
    "الرَّحْمَٰنِ الرَّحِيمِ",
    "مَالِكِ يَوْمِ الدِّينِ",
    "إِيَّاكَ نَعْبُدُ",
]

for phrase in phrases:
    results = quran.search(phrase, mode=SearchMode.DIACRITICS_INSENSITIVE)
    print(f"'{phrase}' — {len(results)} results")
    for r in results:
        print(f"  {r.reference}: [{r.match_type.value}]")
    print()

## Word Occurrence Search

Find all occurrences of a specific word form.

In [ ]:
# Search for specific words
words = ["اللَّهُ", "الرَّحْمَٰنِ", "الرَّحِيمِ", "مَلِك", "عَالَم"]

for word in words:
    results = quran.search(word, mode=SearchMode.SEARCH_BROAD)
    print(f"'{word}' (broad) — {len(results)} occurrences")
    for r in results[:5]:
        print(f"  {r.reference}")
    if len(results) > 5:
        print(f"  ... and {len(results) - 5} more")
    print()

## Register Custom Normalization Profile

Create a custom profile for specific research needs.

In [ ]:
from qurankit.search import NormalizationProfile, register_profile

# Custom profile: keep only base letters, remove everything else
custom = NormalizationProfile(
    profile_id="base-letters-only-v1",
    description="Keep only base Arabic letters, remove all diacritics and marks",
    remove_diacritics=True,
    remove_quranic_marks=True,
    remove_tatweel=True,
    alif_hamza_equivalence=True,
    ya_maqsurah_equivalence=True,
    ta_marbutah_to_ha=True,
)
register_profile(custom)

test = "بِسْمِ اللَّهِ الرَّحْمَٰنِ الرَّحِيمِ۞"
result = engine.normalize(test, "base-letters-only-v1")
print(f"Custom profile result: {result.normalized}")

# Verify it's in the list
profiles = engine.list_profiles()
print(f"Total profiles: {len(profiles)}")
for p in profiles:
    print(f"  {p.profile_id}")

## Search with Context

Get surrounding ayahs for search results.

In [ ]:
# Search and show context
results = quran.search("الْحَمْدُ", mode=SearchMode.DIACRITICS_INSENSITIVE)

for r in results[:3]:
    print(f"Match: {r.reference} - {r.matched_text}")
    # Get surrounding ayahs
    surah_num, ayah_num = map(int, r.reference.split(":"))
    context = quran.get_ayahs_range(surah_num, max(1, ayah_num-1), surah_num, ayah_num+1)
    for a in context:
        marker = ">>> " if a.number == ayah_num else "    "
        print(f"  {marker}{a.reference}: {a.text_original[:80]}")
    print()